In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import time
import anthropic
import os 


api_key = os.environ.get("ANTHROPIC_API_KEY")

vsechny = []

hledat = "stolek"
rubriky = "nabytek"
lokalita = "10100"
humkreis = 10
cenaod = 300
cenado = 1000
final_path = r"/Users/Sam/Downloads/scraping/stůl.xlsx"
# dostane se na stránku
for i in range (0, 100, 20):
    url = f"https://nabytek.bazos.cz/{i}/?hledat={hledat}&rubriky={rubriky}&hlokalita={lokalita}&humkreis={humkreis}&cenaod={cenaod}&cenado={cenado}"
    response = requests.get(url)
    soup = BeautifulSoup(response.text,"lxml")
    inzeraty = soup.find_all("div", class_="inzeraty")

# vytváření df z inzerátů na první stránce
    for div in inzeraty: 
        nadpis = div.find("h2", class_="nadpis")
        if nadpis: 
            zaznam = {
                "titulek": nadpis.text, 
                "url": nadpis.find("a")["href"],
                "cena": div.find("div", class_="inzeratycena").text,
                "lokalita": div.find("div", class_="inzeratylok").get_text(separator="|"),
                "popis": div.find("div", class_="popis").text,
                "foto": div.find("img")["src"],
                "datum_pridani": div.find("span", class_="velikost10").text
            }
            vsechny.append(zaznam)

    print(f"Stránka {i // 20+1}: staženo {len(inzeraty)} inzerátů")
    time.sleep(1)

df = pd.DataFrame(vsechny)
print(f"\nCelkem: {len(df)} inzerátů")
df.head()

# df.to_excel(final_path, index=False)

df["cena_cislo"] = df["cena"].str.replace("Kč", "").str.replace(" ", "").str.strip()
df["cena_cislo"] = df["cena_cislo"].astype(int)
df["mesto"] = df["lokalita"].str.split("|").str[0].str.strip()
df["psc"] = df["lokalita"].str.split("|").str[1].str.strip()
df["full_url"] = "https://nabytek.bazos.cz" + df["url"]

session = requests.Session()
session.cookies.set("bid", "85664201", domain=".bazos.cz")
session.cookies.set("bkod", "HFLSNZ1ZXJ", domain=".bazos.cz")
session.cookies.set("bjmeno", "Samuel", domain=".bazos.cz")
session.cookies.set("btelefon", "777006248", domain=".bazos.cz")
session.cookies.set("testcookie", "ano", domain=".bazos.cz")

detail_data = []

for url in df["full_url"]:
    response = session.get(url)
    soup = BeautifulSoup(response.text, "lxml")

    # popis
    popis_elem = soup.find("div", class_="popisdetail")
    popis = popis_elem.text if popis_elem else None

    # fotky
    fotky = soup.find_all("img", class_="carousel-cell-image")
    foto_urls = [img["data-flickity-lazyload"] for img in fotky]

    # # telefon - extract idi and idphone from onclick
    # tel_span = soup.find("span", class_="teldetail")
    # tel = None
    # if tel_span and tel_span.get("onclick"):
    #     onclick = tel_span["onclick"]
    #     params = onclick.split("'")[3]
    #     idi = params.split("&")[0].split("=")[1]
    #     idphone = params.split("&")[1].split("=")[1]
    #     tel_response = session.post(
    #         "https://nabytek.bazos.cz/ad-phone.php",
    #         data={"idi": idi, "idphone": idphone, "teloverit": "777006248"}
    #     )
    #     tel_soup = BeautifulSoup(tel_response.text, "lxml")
    #     tel_elem = tel_soup.find("a", class_="teldetail")
    #     tel = tel_elem.text if tel_elem else None

    detail_data.append({
        "full_url": url,
        "popis_detail": popis,
        "foto_urls": foto_urls,
        "pocet_fotek": len(foto_urls)
       # "telefon": tel
    })

    print(f"Staženo: {url[-40:]} | tel: {tel}")
    time.sleep(1)

df_detail = pd.DataFrame(detail_data)
print(f"\nStaženo detailů: {len(df_detail)}")
df_final = df.merge(df_detail, on="full_url", how="left")
print(df_final.columns.tolist())
df_final.head(2)



In [17]:
klicova_slova = ["ton", "masiv", "dub", "retro", "vintage", "renovace", "jitona", "brusel", "starší", "renovaci", "pěkný", "zachovalý"]
max_cena = 2000

# price filter
df_filtered = df_final[df_final["cena_cislo"] <= max_cena]

# keyword score - count how many keywords appear in title + description
def pocet_shod(row):
    text = (row["titulek"] + " " + str(row["popis_detail"])).lower()
    return sum(1 for slovo in klicova_slova if slovo in text)

df_filtered["keyword_score"] = df_filtered.apply(pocet_shod, axis=1)
df_filtered = df_filtered.sort_values("keyword_score", ascending=False)

print(df_filtered[["titulek", "cena_cislo", "keyword_score"]].head(10))

                                    titulek  cena_cislo  keyword_score
85                  Stolek Brusel Lovbacken        1000              2
55          Konferenční stolek masiv ořech,         599              2
53                       Konferenční stolek         600              2
48  Retro TV stolek s otočnou deskou Brusel         690              2
8           Stolek scandi Brusel 1950s styl         399              2
77                             Stolek na PC         450              2
74          Konferenční stolek na kolečkách         400              1
22                 Konferenční stůl Mazzoni         999              1
25                   Masivní venkovní židle         800              1
26                 Kulatý stolek k renovaci         499              1


In [ ]:
top = df_filtered.head(5)
scores = []

for idx, row in top.iterrows():
    # Download first photo
    foto_url = row["foto_urls"][0] if row["foto_urls"] else None
    
    content = []
    if foto_url:
        img_response = requests.get(foto_url)
        img_base64 = base64.b64encode(img_response.content).decode("utf-8")
        content.append({"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": img_base64}})
    
    content.append({"type": "text", "text": f"""Ohodnoť tento inzerát na stupnici 1-10 pro nákup a další prodej nábytku.
Cíl: zisk minimálně 1000 Kč na kusu po renovaci a všech nákladech.
Kritéria: stav, cena, potenciál zisku, lokace (Praha = lepší), vizuální stav na fotce.

Titulek: {row['titulek']}
Cena: {row['cena']}
Město: {row['mesto']}
Popis: {row['popis_detail']}

Odpověz POUZE ve formátu:
SCORE: X/10
DŮVOD: (krátké vysvětlení)
STAV DLE FOTKY: (co vidíš)"""})

    message = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=300,
        messages=[{"role": "user", "content": content}]
    )
    
    ai_text = message.content[0].text
    scores.append({"full_url": row["full_url"], "ai_hodnoceni": ai_text})
    print(f"\n{row['titulek']} | {row['cena']}")
    print(ai_text)
    time.sleep(1)

df_scores = pd.DataFrame(scores)

In [8]:
import pandas as pd

kriteria_excel = r"/Users/Sam/scripts/kriteria.xlsx"

df_kriteria = pd.read_excel(kriteria_excel, "kriteria")
df_keywords = pd.read_excel(kriteria_excel, "keywords")

print(df_keywords)
print(df_kriteria)

   hledani                                      klicova_slova
0      NaN  ton, retro, vintage, masiv, jitona, jiroutek, ...
  hledat  rubriky  lokalita  humkreis  cena od  cena do
0   stůl  nabytek     10100        10        0     1500
1  židle  nabytek     10100        10        0     1500


In [ ]:
for i in range(0, 100, 20):
    url = f"https://nabytek.bazos.cz/{i}/?hledat={hledat}&rubriky={rubriky}&hlokalita={lokalita}&humkreis={humkreis}&cenaod={cenaod}&cenado={cenado}"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "lxml")
    inzeraty = soup.find_all("div", class_="inzeraty")

    for div in inzeraty:
        nadpis = div.find("h2", class_="nadpis")
        if nadpis:
            zaznam = {
                "titulek": nadpis.text,
                "url": nadpis.find("a")["href"],
                "cena": div.find("div", class_="inzeratycena").text,
                "lokalita": div.find("div", class_="inzeratylok").get_text(separator="|"),
                "popis": div.find("div", class_="popis").text,
                "foto": div.find("img")["src"],
                "datum_pridani": div.find("span", class_="velikost10").text
            }
            vsechny.append(zaznam)

    print(f"Stránka {i // 20 + 1}: staženo {len(inzeraty)} inzerátů")
    time.sleep(1)

df = pd.DataFrame(vsechny)
print(f"\nCelkem: {len(df)} inzerátů")

In [9]:
kriteria_excel = r"/Users/Sam/scripts/kriteria.xlsx"

df_kriteria = pd.read_excel(kriteria_excel, "kriteria")

for idx, row in df_kriteria.iterrows():
    hledat = row["hledat"]
    rubriky = row["rubriky"]
    lokalita = row["lokalita"]
    humkreis = row["humkreis"]
    cenaod = row["cena od"]
    cenado = row["cena do"]
    klicova_slova = row["klicova_slova"].split(", ")


KeyError: 'klicova_slova'

In [ ]:
df_scores["score"] = df_scores["ai_hodnoceni"].str.extract(r"SCORE: (\d+)/10")
df_scores["score"] = df_scores["score"].astype(int)
df_scores = df_scores.sort_values("score", ascending=False)
print(df_scores[["full_url", "score"]])

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import anthropic
import base64
import time
import os

# ==================== NASTAVENÍ ====================
hledat = "židle ton"
rubriky = "nabytek"
lokalita = "10100"
humkreis = 25
cenaod = 300
cenado = 1000
max_cena = 2000
klicova_slova = ["ton", "masiv", "dub", "retro", "vintage", "renovace"]
top_n = 5
final_path = r"/Users/Sam/Downloads/scraping/bazos_ranked.xlsx"

# ==================== COOKIES ====================
session = requests.Session()
session.cookies.set("bid", "85664201", domain=".bazos.cz")
session.cookies.set("bkod", "HFLSNZ1ZXJ", domain=".bazos.cz")
session.cookies.set("bjmeno", "Samuel", domain=".bazos.cz")
session.cookies.set("btelefon", "777006248", domain=".bazos.cz")
session.cookies.set("testcookie", "ano", domain=".bazos.cz")

# ==================== API ====================
os.environ["ANTHROPIC_API_KEY"] = "your-key-here"
client = anthropic.Anthropic()

# ==================== 1. SCRAPING ====================
vsechny = []

for i in range(0, 100, 20):
    url = f"https://nabytek.bazos.cz/{i}/?hledat={hledat}&rubriky={rubriky}&hlokalita={lokalita}&humkreis={humkreis}&cenaod={cenaod}&cenado={cenado}"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "lxml")
    inzeraty = soup.find_all("div", class_="inzeraty")

    for div in inzeraty:
        nadpis = div.find("h2", class_="nadpis")
        if nadpis:
            zaznam = {
                "titulek": nadpis.text,
                "url": nadpis.find("a")["href"],
                "cena": div.find("div", class_="inzeratycena").text,
                "lokalita": div.find("div", class_="inzeratylok").get_text(separator="|"),
                "popis": div.find("div", class_="popis").text,
                "foto": div.find("img")["src"],
                "datum_pridani": div.find("span", class_="velikost10").text
            }
            vsechny.append(zaznam)

    print(f"Stránka {i // 20 + 1}: staženo {len(inzeraty)} inzerátů")
    time.sleep(1)

df = pd.DataFrame(vsechny)
print(f"\nCelkem: {len(df)} inzerátů")

# ==================== 2. ČIŠTĚNÍ DAT ====================
df["cena_cislo"] = df["cena"].str.replace("Kč", "").str.replace(" ", "").str.strip()
df["cena_cislo"] = df["cena_cislo"].astype(int)
df["mesto"] = df["lokalita"].str.split("|").str[0].str.strip()
df["psc"] = df["lokalita"].str.split("|").str[1].str.strip()
df["full_url"] = "https://nabytek.bazos.cz" + df["url"]

# ==================== 3. DETAILY ====================
detail_data = []

for url in df["full_url"]:
    response = session.get(url)
    soup = BeautifulSoup(response.text, "lxml")

    popis_elem = soup.find("div", class_="popisdetail")
    popis = popis_elem.text if popis_elem else None

    fotky = soup.find_all("img", class_="carousel-cell-image")
    foto_urls = [img["data-flickity-lazyload"] for img in fotky]

    detail_data.append({
        "full_url": url,
        "popis_detail": popis,
        "foto_urls": foto_urls,
        "pocet_fotek": len(foto_urls)
    })

    print(f"Detail: {url[-40:]}")
    time.sleep(1)

df_detail = pd.DataFrame(detail_data)
df = df.merge(df_detail, on="full_url", how="left")

# ==================== 4. RULE-BASED FILTR ====================
df_filtered = df[df["cena_cislo"] <= max_cena].copy()

def pocet_shod(row):
    text = (row["titulek"] + " " + str(row["popis_detail"])).lower()
    return sum(1 for slovo in klicova_slova if slovo in text)

df_filtered["keyword_score"] = df_filtered.apply(pocet_shod, axis=1)
df_filtered = df_filtered.sort_values("keyword_score", ascending=False)
print(f"\nProšlo filtrem: {len(df_filtered)} inzerátů")

# ==================== 5. AI SCORING ====================
top = df_filtered.head(top_n)
scores = []

for idx, row in top.iterrows():
    foto_url = row["foto_urls"][0] if row["foto_urls"] else None

    content = []
    if foto_url:
        img_response = requests.get(foto_url)
        img_base64 = base64.b64encode(img_response.content).decode("utf-8")
        content.append({"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": img_base64}})

    content.append({"type": "text", "text": f"""Ohodnoť tento inzerát na stupnici 1-10 pro nákup a další prodej nábytku.
Cíl: zisk minimálně 1000 Kč na kusu po renovaci a všech nákladech.
Kritéria: stav, cena, potenciál zisku, lokace (Praha = lepší), vizuální stav na fotce.

Titulek: {row['titulek']}
Cena: {row['cena']}
Město: {row['mesto']}
Popis: {row['popis_detail']}

Odpověz POUZE ve formátu:
SCORE: X/10
DŮVOD: (krátké vysvětlení)
STAV DLE FOTKY: (co vidíš)"""})

    message = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=300,
        messages=[{"role": "user", "content": content}]
    )

    ai_text = message.content[0].text
    scores.append({"full_url": row["full_url"], "ai_hodnoceni": ai_text})
    print(f"\n{row['titulek']} | {row['cena']}")
    print(ai_text)
    time.sleep(1)

df_scores = pd.DataFrame(scores)
df_scores["score"] = df_scores["ai_hodnoceni"].str.extract(r"SCORE: (\d+)/10")
df_scores["score"] = df_scores["score"].astype(int)

df_ranked = df_filtered.merge(df_scores, on="full_url", how="left")
df_ranked = df_ranked.sort_values("score", ascending=False)

# ==================== 6. TELEFONY (TOP 3) ====================
for idx, row in df_ranked.head(3).iterrows():
    response = session.get(row["full_url"])
    soup = BeautifulSoup(response.text, "lxml")

    tel_span = soup.find("span", class_="teldetail")
    if tel_span and tel_span.get("onclick"):
        onclick = tel_span["onclick"]
        params = onclick.split("'")[3]
        idi = params.split("&")[0].split("=")[1]
        idphone = params.split("&")[1].split("=")[1]
        tel_response = session.post(
            "https://nabytek.bazos.cz/ad-phone.php",
            data={"idi": idi, "idphone": idphone, "teloverit": "777006248"}
        )
        tel_soup = BeautifulSoup(tel_response.text, "lxml")
        tel_elem = tel_soup.find("a", class_="teldetail")
        tel = tel_elem.text if tel_elem else "N/A"
    else:
        tel = "N/A"

    df_ranked.loc[idx, "telefon"] = tel
    print(f"Tel pro {row['titulek']}: {tel}")
    time.sleep(3)

# ==================== 7. EXPORT ====================
df_ranked.to_excel(final_path, index=False)
print(f"\nHotovo! Uloženo do: {final_path}")